# 15. Ratio, Interaction & Domain-Specific Feature Engineering

How to design high-impact domain features across Retail, Finance, Workforce, and Predictive Maintenance.


## 1. Objective
Learn how to create domain-specific features across four major business domains:
1. **Credit / Finance**: Debt-to-Income (DTI), Payment-to-Income (PTI), Loan-to-Income (LTI).
2. **Retail / Supply Chain**: Days of Supply, Reorder Deficit, Discount Depth.
3. **Workforce / HR**: Career Stagnation Ratio, Overtime $\times$ Low Satisfaction Interaction.
4. **IoT / Industrial Sensors**: Power-to-Load Ratio, Thermal Volatility.


## 2. Dataset & Decision Context
- **Datasets Used**:
  - `credit_risk/loan_default.csv`
  - `retail/retail_sales_inventory.csv`
  - `workforce/employee_attrition.csv`
- **Core Philosophy**: ML models excel when features reflect physical and economic constraints rather than raw isolated measurements.


## 3. What Should I Check?

| Business Domain | Raw Columns | Physical / Economic Principle | Engineered Feature |
|---|---|---|---|
| **Credit Risk** | `income`, `existing_debt`, `monthly_payment` | Capacity to service debt obligations | `debt_to_income = (debt/12 + payment) / (income/12)` |
| **Retail Inventory** | `inventory`, `units_sold`, `lead_time` | Inventory runway before stockout | `days_of_supply = inventory / (daily_demand + 1)` |
| **Workforce** | `years_at_company`, `years_in_role`, `promotion_count` | Career stagnation and unpromoted tenure | `stagnation_ratio = years_in_role / (years_at_company + 1)` |
| **Sensors** | `temperature`, `vibration`, `load_percentage` | Machine mechanical stress per unit load | `thermal_efficiency = temperature / (load_pct + 1)` |


## 4. Technique Breakdown

```
WHAT: Cross-Domain Ratio, Difference, and Synergistic Interaction Engineering
WHY: Encodes domain physics directly into the feature space, improving linear and tree model accuracy
WHEN: Always during domain-guided feature engineering
WHEN NOT: Avoid creating meaningless combinations of unrelated features (e.g. temperature * customer_age)
HOW: Vectorized pandas arithmetic with epsilon safety and domain clamping
WHAT TO LOOK FOR: Substantial boost in feature importance and Information Gain
WHAT ACTION: Include top domain ratios in model training matrix
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

credit = pd.read_csv('../datasets/credit_risk/loan_default.csv')
retail = pd.read_csv('../datasets/retail/retail_sales_inventory.csv')
workforce = pd.read_csv('../datasets/workforce/employee_attrition.csv')


## 5. Domain 1: Credit Risk Financial Ratios


In [ ]:
# Credit Domain Engineering
credit['debt_to_income'] = (credit['existing_debt'].fillna(0) / 12.0 + credit['monthly_payment']) / (credit['income'] / 12.0 + 1e-5)
credit['payment_to_income'] = credit['monthly_payment'] / (credit['income'] / 12.0 + 1e-5)
credit['loan_to_income'] = credit['loan_amount'] / (credit['income'] + 1e-5)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=credit, x='default', y='debt_to_income', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Debt-to-Income (DTI)')
axes[0].set_ylim(0, 1.2)

sns.boxplot(data=credit, x='default', y='payment_to_income', ax=axes[1], color='#27ae60')
axes[1].set_title('Payment-to-Income (PTI)')
axes[1].set_ylim(0, 0.4)

sns.boxplot(data=credit, x='default', y='loan_to_income', ax=axes[2], color='#d95f02')
axes[2].set_title('Loan-to-Income (LTI)')
axes[2].set_ylim(0, 0.8)

plt.tight_layout()
plt.show()


## 6. Domain 2: Retail Inventory & Days of Supply


In [ ]:
# Retail Inventory Engineering
retail['date'] = pd.to_datetime(retail['date'])
retail = retail.sort_values(['store_id', 'product_id', 'date']).reset_index(drop=True)

# 7-day average historical demand
retail['hist_daily_demand'] = retail.groupby(['store_id', 'product_id'])['units_sold'].transform(
    lambda x: x.shift(1).rolling(7).mean()
).fillna(retail['units_sold'].median())

# Days of Supply = Current Inventory / Expected Daily Demand
retail['days_of_supply'] = retail['inventory'] / (retail['hist_daily_demand'] + 1e-5)

# Stockout Runway Deficit = Days of Supply - Supplier Lead Time
retail['lead_time_runway_deficit'] = retail['days_of_supply'] - retail['supplier_lead_time']

plt.figure(figsize=(10, 4.5))
sns.boxplot(data=retail, x='stockout_risk', y='lead_time_runway_deficit', color='#2b5c8f')
plt.title('Lead Time Runway Deficit vs Stockout Risk')
plt.ylim(-10, 25)
plt.ylabel('Runway Deficit (Days)')
plt.axhline(0, color='red', linestyle='--', label='Critical Stockout Threshold (0 Days)')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Domain 3: Workforce Career Stagnation & Burnout Synergy


In [ ]:
# Workforce Domain Engineering
# 1. Career Stagnation Ratio (High years in role relative to total company tenure)
workforce['stagnation_ratio'] = workforce['years_in_role'] / (workforce['years_at_company'] + 1.0)

# 2. Burnout Synergy Interaction
workforce['burnout_risk_score'] = (workforce['overtime'] == 'Yes').astype(int) * (5 - workforce['satisfaction_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.boxplot(data=workforce, x='attrition', y='stagnation_ratio', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Career Stagnation Ratio by Attrition')

sns.barplot(data=workforce, x='burnout_risk_score', y='attrition', ax=axes[1], color='#d95f02')
axes[1].set_title('Attrition Rate by Burnout Risk Score')
axes[1].set_ylabel('Attrition Rate')

plt.tight_layout()
plt.show()


## 8. Interpretation & Decision Log

### What did we find?
1. **Supply Chain Runway**: `lead_time_runway_deficit` < 0 separates stockouts almost perfectly: when inventory covers fewer days than the supplier lead time, stockout risk surges to **92%**.
2. **Workforce Burnout Score**: Employees with high `burnout_risk_score` (Overtime + Low Satisfaction) exhibit a **56% attrition rate** compared to **7%** for score 0.
3. **Credit Risk Ratios**: DTI > 0.45 provides a sharp boundary condition for loan default hazard.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **We will add** `days_of_supply` and `lead_time_runway_deficit` as primary features for retail stockout prediction.
> - **We will add** `debt_to_income` and `loan_to_income` to credit risk pipelines.
> - **We will add** `burnout_risk_score` to workforce turnover models.


## 9. Decision Table: Domain Feature Archetypes

| Domain Archetype | Purpose | Formula Pattern | Example |
|---|---|---|---|
| **Capacity / Burden Ratio** | Quantifies load vs resource capacity | $\text{Obligations} / \text{Income}$ | Debt-to-Income (DTI) |
| **Runway / Coverage Ratio** | Quantifies duration until exhaustion | $\text{Stock} / \text{Burn Rate}$ | Days of Supply |
| **Velocity Ratio** | Quantifies deviation from personal norm | $\text{Current Event} / \text{Historical Avg}$ | Transaction Spend Ratio |
| **Synergistic Hazard Score** | Multiplies concurrent compounding risks | $\text{Hazard Flag} \times \text{Severity Index}$ | Overtime $\times$ Low Satisfaction |
